# 概率、校准与回退

这是本站的**合成数据教学实验**，不调用 Jev 或任何模型。无需密钥、安装包或 GPU。

任务：先运行，再修改一个高置信度错误样本，保存观察和解释。

In [ ]:
labels = [1, 0, 1, 0, 1, 0, 1, 0]
probabilities = [0.95, 0.10, 0.80, 0.85, 0.60, 0.40, 0.55, 0.20]
assert len(labels) == len(probabilities)

def brier(labels, probabilities):
    return sum((p-y)**2 for y, p in zip(labels, probabilities))/len(labels)

def selective(labels, probabilities, threshold):
    # 此处使用二元预测的最大类别概率；不是 TypeSafe confidence 公式。
    kept = [(y, p) for y,p in zip(labels, probabilities) if max(p,1-p) >= threshold]
    errors = sum((p >= 0.5) != bool(y) for y,p in kept)
    return {"threshold":threshold, "accepted":len(kept), "errors":errors, "coverage":len(kept)/len(labels), "accepted_error":errors/len(kept) if kept else None}

print("Brier:", round(brier(labels, probabilities), 4))
for threshold in [0.6,0.9]:
    print(selective(labels, probabilities, threshold))

## 只改变一个变量

下面把第一个原本正确的高置信度样本标签翻转。两次使用相同预测，观察 Brier 和自动处理子集的错误率如何变化。小样本没有统计结论。

In [ ]:
changed = labels.copy()
changed[0] = 0
print("Changed Brier:", round(brier(changed, probabilities), 4))
print(selective(changed, probabilities, 0.9))
assert brier(changed, probabilities) > brier(labels, probabilities)
assert selective(changed, probabilities, 0.9)["accepted_error"] > 0

## 写下你的理解

1. 高概率为什么仍会答错？
2. 阈值升高时覆盖率发生了什么？
3. 如果低置信度请求交给另一个模型，如何计算整个系统成本与时延？
4. 真实评价需要验证集选阈值、独立测试集验收；不要用这八个样本评价 Jev。

回到 [学习单元](https://suguilin.github.io/study/?id=digest-jev#study-practice)，在第三题写下解释；运行最后一格，下载 `calibration_report.json`，用阅读页「导入实验结果」保存指标到笔记。

原始来源：[TypeSafe confidence](https://docs.typesafe.ai/confidence)、[独立评测摘要](https://arxiv.org/abs/2609.37647)。

## 两种阈值不要混用

本实验用二元最大类别概率 `max(p, 1-p)` 筛选样本。[官方 Choice 公式](https://docs.typesafe.ai/confidence#choice)在两个选项时为 `(p_max - 0.5) / 0.5`。所以概率阈值 0.9 对应二元 Choice confidence 0.8；不是同一个数字。Score 的公式不同，不能照搬。

这是公式对照，不代表预测已校准，也没有调用 Jev。


In [ ]:
def binary_choice_confidence(top_probability):
    assert 0.5 <= top_probability <= 1
    return (top_probability - 0.5) / 0.5

print("Binary Choice confidence for probability threshold 0.9:", binary_choice_confidence(0.9))
assert abs(binary_choice_confidence(0.9) - 0.8) < 1e-12


## 保存实验与解释

补充下面的 `my_interpretation` 后运行。Colab 文件侧栏会出现 `calibration_report.json`，下载后导入本篇笔记；再存入学习日志或结束这次学习即可保留快照。

如果修改了实验步骤，同时修改 `change_description`。报告只保存汇总指标与解释，不保存逐条样本。


In [ ]:
from pathlib import Path
import json

my_interpretation = ""  # 写下观察、假设与下一步验证。
change_description = "保持预测不变，将第一个样本标签从 1 改为 0"
thresholds = [0.6, 0.9]
report = {
    "experiment": "decision-calibration",
    "data_scope": "synthetic binary classification",
    "model_evaluated": False,
    "samples": len(labels),
    "change": change_description,
    "baseline": {"brier": brier(labels, probabilities),
                 "selective": [selective(labels, probabilities, t) for t in thresholds]},
    "changed": {"brier": brier(changed, probabilities),
                "selective": [selective(changed, probabilities, t) for t in thresholds]},
    "my_interpretation": my_interpretation,
}
Path("calibration_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved calibration_report.json. 回到专题阅读页导入实验结果。")
